# Scratch: Create 7-Band WAC Dataset

Creates `data_7band/{split}/{chips,labels}` from `data/{split}/{chips,labels}`. Chip TIFFs keep only bands 1-7. Labels are copied unchanged.

In [1]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import logging
import os
import shutil

try:
    import pyproj
    os.environ["PROJ_LIB"] = pyproj.datadir.get_data_dir()
except Exception:
    pass

logging.getLogger("rasterio._env").setLevel(logging.ERROR)

import rasterio

SRC_ROOT = Path("data")
DST_ROOT = Path("data_7band")
MAX_WORKERS = 16


def write_7band_chip(args):
    chip_path, out_path = args
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")

        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)

    return str(out_path)


def copy_label(args):
    label_path, out_path = args
    out_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, out_path)
    return str(out_path)


def process_split(split):
    src_chips = SRC_ROOT / split / "chips"
    src_labels = SRC_ROOT / split / "labels"
    dst_chips = DST_ROOT / split / "chips"
    dst_labels = DST_ROOT / split / "labels"

    chip_jobs = [(p, dst_chips / p.name) for p in sorted(src_chips.glob("*.tif"))]
    label_jobs = [(p, dst_labels / p.name) for p in sorted(src_labels.iterdir()) if p.is_file()]

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        chip_outputs = list(executor.map(write_7band_chip, chip_jobs))

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        label_outputs = list(executor.map(copy_label, label_jobs))

    print(f"{split}: wrote {len(chip_outputs)} chips and copied {len(label_outputs)} labels")

In [2]:
# for split in ["train"]:
#     process_split(split)

In [3]:
# for split in ["val"]:
#     process_split(split)

In [4]:
# for split in ["test"]:
#     process_split(split)

## Create Semantic Segmentation Splits

Builds a fresh `train`/`val`/`test` split from a semantic-segmentation source directory. This does not use the existing `data` split layout.

In [5]:
import random

LFM_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs")
SEM_SEG_ROOT = LFM_ROOT / "7_band_vis_uv/sem_seg"
SEM_SEG_OUTPUT_ROOT = LFM_ROOT / "full_model_sem_seg_v2"

SEM_SEG_CHIPS_DIR = SEM_SEG_ROOT / "chips"
SEM_SEG_LABELS_DIR = SEM_SEG_ROOT / "labels"

SEM_SEG_IMAGE_GLOB = "*.tif"
SEM_SEG_LABEL_GLOB = "*_label.*"
SEM_SEG_IMAGE_SUFFIX = "_input_wac_static_chip"
SEM_SEG_LABEL_SUFFIX = "_label"

SEM_SEG_SEED = 42
SEM_SEG_N_TEST = 100
SEM_SEG_TRAIN_FRACTION = 0.95


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_sem_seg_pairs():
    chips = {
        split_key(path, SEM_SEG_IMAGE_SUFFIX): path
        for path in sorted(SEM_SEG_CHIPS_DIR.glob(SEM_SEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, SEM_SEG_LABEL_SUFFIX): path
        for path in sorted(SEM_SEG_LABELS_DIR.glob(SEM_SEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= SEM_SEG_N_TEST:
        raise ValueError(f"Need more than {SEM_SEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def copy_sem_seg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = SEM_SEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = SEM_SEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_7band_chip((chip_path, chip_out))
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)
    return split


def create_sem_seg_split():
    pairs = find_sem_seg_pairs()
    rng = random.Random(SEM_SEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:SEM_SEG_N_TEST]
    remaining = pairs[SEM_SEG_N_TEST:]
    n_train = int(round(len(remaining) * SEM_SEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        list(executor.map(copy_sem_seg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {SEM_SEG_OUTPUT_ROOT.resolve()}")

In [6]:
create_sem_seg_split()

matched pairs: 623
chips only:    0
labels only:   51
train: 497 pairs
val: 26 pairs
test: 100 pairs
wrote split dataset to: /panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_sem_seg_v2
